In [1]:
import pandas as pd
import re
import string
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [3]:
fake = pd.read_csv("fake.csv")
real_news = pd.read_csv("real_news.csv")

In [4]:
fake.head()

,label,text
0,0,Breaking: Secret government project exposed ca...
1,0,Breaking: Secret government project exposed ca...
2,0,Breaking: Secret government project exposed ca...
3,0,Breaking: Secret government project exposed ca...
4,0,Breaking: Secret government project exposed ca...


In [5]:
real_news.head()

,label,text
0,1,Report: Government releases official statement...
1,1,Report: Government releases official statement...
2,1,Report: Government releases official statement...
3,1,Report: Government releases official statement...
4,1,Report: Government releases official statement...


In [7]:
fake['class']=0
real_news['class']=1

In [8]:
data=pd.concat([fake,real_news],axis=0)

In [9]:
data.sample(10)

,label,text,class
474,1,Report: Government releases official statement...,1
866,0,Breaking: Secret government project exposed ca...,0
85,1,Report: Government releases official statement...,1
622,0,Breaking: Secret government project exposed ca...,0
780,0,Breaking: Secret government project exposed ca...,0
951,1,Report: Government releases official statement...,1
413,0,Breaking: Secret government project exposed ca...,0
77,0,Breaking: Secret government project exposed ca...,0
846,0,Breaking: Secret government project exposed ca...,0
354,0,Breaking: Secret government project exposed ca...,0


In [14]:
data = data.drop(columns=["title", "subject", "date"], axis=1, errors="ignore")


In [15]:
data.reset_index(inplace=True)

In [16]:
data.drop(["index"],axis=1,inplace=True)

In [17]:
data.sample(5)

,label,text,class
144,0,Breaking: Secret government project exposed ca...,0
1895,1,Report: Government releases official statement...,1
151,0,Breaking: Secret government project exposed ca...,0
1594,1,Report: Government releases official statement...,1
658,0,Breaking: Secret government project exposed ca...,0


In [18]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)                # remove text in brackets
    text = re.sub(r'\W', ' ', text)                    # remove non-word characters
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'<.*?>+', '', text)                 # remove HTML tags
    text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)
    text = re.sub(r'\n', ' ', text)                    # remove new lines
    text = re.sub(r'\w*\d\w*', '', text)               # remove words with numbers
    return text


In [19]:
data["text"]=data["text"].apply(clean_text)

In [23]:
x=data["text"]
y=data["text"]

In [24]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.25, random_state=42)

In [25]:
vectorizer=TfidfVectorizer()
xy_train=vectorizer.fit_transform(xtrain)
xy_test=vectorizer.transform(xtest)

In [26]:
lr=LogisticRegression()
lr.fit(xy_train,ytrain)

LogisticRegression()

In [27]:
prediction=lr.predict(xy_test)
lr.score(xy_test,ytest)

1.0

In [28]:
print(classification_report(ytest,prediction))

                                                                                                                                                       precision    recall  f1-score   support

  breaking  secret government project exposed causing widespread panic among citizens number   experts claim shocking hidden agenda without evidence        1.00      1.00      1.00       257
report  government releases official statement on economic development plan number   highlighting infrastructure growth and employment opportunities        1.00      1.00      1.00       243

                                                                                                                                             accuracy                           1.00       500
                                                                                                                                            macro avg       1.00      1.00      1.00       500
                                           

In [29]:
joblib.dump(vectorizer,"vectorizer.jb")
joblib.dump(lr,"lr_model.jb")

['lr_model.jb']